# Lesson 3: Touring the Listings - Selecting, Filtering & Transforming

---

## The Story

Imagine you are a buyer's agent. Your client has a wishlist:

- At least 3 bedrooms
- Price between $300,000 and $700,000
- Preferably in Riverside or Downtown
- Wants to know the price per square foot for each option

You have 510 listings. Time to **tour** them -- selecting only the columns that matter,
filtering out the ones that do not fit, and computing new derived values to help your client decide.

---

## What You Will Learn

- `select()` -- choose specific columns from a DataFrame
- `filter()` / `where()` -- keep only rows matching your criteria
- `col()` and `alias()` -- reference columns programmatically and rename them
- `withColumn()` -- add new computed columns
- `orderBy()` / `sort()` -- sort your results
- `distinct()` / `dropDuplicates()` -- remove duplicate rows
- Chaining multiple operations into a clean pipeline


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, round as spark_round
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, DoubleType, FloatType
)

spark = SparkSession.builder \
    .appName("TouringTheListings") \
    .master("local[*]") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

listings_schema = StructType([
    StructField("property_id",   StringType(),  True),
    StructField("address",       StringType(),  True),
    StructField("city",          StringType(),  True),
    StructField("zip_code",      StringType(),  True),
    StructField("property_type", StringType(),  True),
    StructField("bedrooms",      IntegerType(), True),
    StructField("bathrooms",     FloatType(),   True),
    StructField("sqft",          IntegerType(), True),
    StructField("list_price",    DoubleType(),  True),
    StructField("year_built",    IntegerType(), True),
    StructField("neighborhood",  StringType(),  True),
    StructField("status",        StringType(),  True),
    StructField("listing_date",  StringType(),  True),
    StructField("agent_id",      StringType(),  True)
])

df = spark.read \
    .option("header", True) \
    .schema(listings_schema) \
    .csv("data/property_listings.csv")

print(f"Dataset loaded: {df.count()} listings")
df.show(3)

Dataset loaded: 510 listings


+-----------+-------------+--------+--------+-------------+--------+---------+----+----------+----------+------------+-------+------------+--------+
|property_id|      address|    city|zip_code|property_type|bedrooms|bathrooms|sqft|list_price|year_built|neighborhood| status|listing_date|agent_id|
+-----------+-------------+--------+--------+-------------+--------+---------+----+----------+----------+------------+-------+------------+--------+
|  PROP-0001|7370 River Rd|Parkview|   84038|        House|       5|      1.0|NULL| 1268597.0|      NULL|   Riverside|   Sold|  2025-12-24| AGT-016|
|  PROP-0002|960 Park Blvd|Parkview|   62393|        House|    NULL|      2.0|NULL|  835503.0|      NULL|     Oakwood|   NULL|  2025-10-07| AGT-004|
|  PROP-0003| 5291 Hill Dr|Parkview|   76081|        House|       3|      1.0|NULL|  487283.0|      NULL|    Hillside|Pending|        NULL| AGT-017|
+-----------+-------------+--------+--------+-------------+--------+---------+----+----------+----------+-

## Selecting Columns - Choose Your Rooms

Just as a buyer does not need to inspect every detail on the first tour,
you do not always need every column. `select()` lets you pick exactly what matters.

### Two Ways to Select

```python
# String column names (simple, readable)
df.select("column_a", "column_b")

# col() objects (flexible, supports expressions)
df.select(col("column_a"), col("column_b") * 2)
```

> **Tip:** `select()` does not modify the original DataFrame -- it returns a **new** DataFrame.
> Everything in Spark is immutable.


In [2]:
# Select only the columns relevant for a buyer tour view
tour_view = df.select(
    "property_id",
    "address",
    "neighborhood",
    "list_price",
    "bedrooms",
    "bathrooms",
    "sqft"
)

print("=== Buyer Tour View (selected columns only) ===")
tour_view.show(5)

print(f"Original columns : {len(df.columns)}")
print(f"Selected columns : {len(tour_view.columns)}")

=== Buyer Tour View (selected columns only) ===


+-----------+-------------+------------+----------+--------+---------+----+
|property_id|      address|neighborhood|list_price|bedrooms|bathrooms|sqft|
+-----------+-------------+------------+----------+--------+---------+----+
|  PROP-0001|7370 River Rd|   Riverside| 1268597.0|       5|      1.0|NULL|
|  PROP-0002|960 Park Blvd|     Oakwood|  835503.0|    NULL|      2.0|NULL|
|  PROP-0003| 5291 Hill Dr|    Hillside|  487283.0|       3|      1.0|NULL|
|  PROP-0004| 5834 Oak Ave|    Downtown|  733117.0|       3|      2.5|NULL|
|  PROP-0005|566 Park Blvd|    Downtown|  951151.0|       4|     NULL|NULL|
+-----------+-------------+------------+----------+--------+---------+----+
only showing top 5 rows
Original columns : 14
Selected columns : 7


## Filtering Rows - Your Search Criteria

A buyer would never walk through every property -- they apply filters first.
`filter()` (also aliased as `where()`) keeps only rows that match a condition.

### Combining Conditions

| Operator | Meaning | Note |
|----------|---------|------|
| `&` | AND | Wrap each condition in `()` |
| `\|` | OR | Wrap each condition in `()` |
| `~` | NOT | Negates a condition |

> **Important:** Always wrap each condition in parentheses `()` when combining
> with `&` or `|` due to Python operator precedence.


In [3]:
# Filter 1: Price range
affordable = df.filter(
    (col("list_price") >= 300000) & (col("list_price") <= 700000)
)
print(f"Listings $300k-$700k:               {affordable.count()}")

# Filter 2: Minimum bedrooms
family_homes = df.filter(col("bedrooms") >= 3)
print(f"Listings with 3+ bedrooms:          {family_homes.count()}")

# Filter 3: Specific neighborhood
downtown = df.filter(col("neighborhood") == "Downtown")
print(f"Downtown listings:                  {downtown.count()}")

# Filter 4: Multiple neighborhoods using isin()
preferred = df.filter(
    col("neighborhood").isin(["Downtown", "Riverside", "Midtown"])
)
print(f"Preferred neighborhood listings:    {preferred.count()}")

# Filter 5: Combined buyer criteria
buyer_matches = df.filter(
    (col("list_price") >= 300000) &
    (col("list_price") <= 700000) &
    (col("bedrooms") >= 3) &
    (col("status") == "Active")
)
print(f"\nActive listings matching criteria:  {buyer_matches.count()}")
buyer_matches.select("address", "neighborhood", "list_price", "bedrooms").show(5)

Listings $300k-$700k:               149


Listings with 3+ bedrooms:          342


Downtown listings:                  72


Preferred neighborhood listings:    187



Active listings matching criteria:  0


+-------+------------+----------+--------+
|address|neighborhood|list_price|bedrooms|
+-------+------------+----------+--------+
+-------+------------+----------+--------+



## Renaming & Aliasing Columns

Column names from raw data are often technical or abbreviated.
Clean column names matter when presenting results to stakeholders.

### Two Ways to Rename

```python
# alias() inside select() -- rename on the fly
df.select(col("list_price").alias("asking_price"))

# withColumnRenamed() -- rename without rewriting the whole select
df.withColumnRenamed("list_price", "asking_price")
```


In [4]:
# Rename columns using alias() inside select()
clean_view = df.select(
    col("property_id").alias("id"),
    col("address").alias("street_address"),
    col("list_price").alias("asking_price"),
    col("sqft").alias("square_feet"),
    col("bedrooms").alias("beds"),
    col("bathrooms").alias("baths"),
    col("neighborhood").alias("area")
)

print("=== Renamed Columns - Stakeholder-Friendly View ===")
clean_view.show(5)

# withColumnRenamed for individual columns
renamed_df = df.withColumnRenamed("property_type", "type") \
               .withColumnRenamed("year_built", "built_year")

print("\nColumns after withColumnRenamed:")
print(renamed_df.columns)

=== Renamed Columns - Stakeholder-Friendly View ===


+---------+--------------+------------+-----------+----+-----+---------+
|       id|street_address|asking_price|square_feet|beds|baths|     area|
+---------+--------------+------------+-----------+----+-----+---------+
|PROP-0001| 7370 River Rd|   1268597.0|       NULL|   5|  1.0|Riverside|
|PROP-0002| 960 Park Blvd|    835503.0|       NULL|NULL|  2.0|  Oakwood|
|PROP-0003|  5291 Hill Dr|    487283.0|       NULL|   3|  1.0| Hillside|
|PROP-0004|  5834 Oak Ave|    733117.0|       NULL|   3|  2.5| Downtown|
|PROP-0005| 566 Park Blvd|    951151.0|       NULL|   4| NULL| Downtown|
+---------+--------------+------------+-----------+----+-----+---------+
only showing top 5 rows

Columns after withColumnRenamed:
['property_id', 'address', 'city', 'zip_code', 'type', 'bedrooms', 'bathrooms', 'sqft', 'list_price', 'built_year', 'neighborhood', 'status', 'listing_date', 'agent_id']


## Adding Computed Columns with withColumn

`withColumn(name, expression)` adds a new column based on an expression.
It never modifies existing columns -- it always returns a new DataFrame.

### Common Use Cases

- **Derived metrics** -- price per sqft, price per bedroom
- **Age calculations** -- current year minus year built
- **Categorizations** -- bucket a price into Low / Mid / High
- **String transformations** -- upper/lower case, concatenation

> **Property analogy:** `withColumn` is like adding a new column to your property spreadsheet
> -- you compute it from existing data and it becomes part of your working view.


In [5]:
# Add computed columns to the listings DataFrame
enriched_df = df \
    .withColumn(
        "price_per_sqft",
        spark_round(col("list_price") / col("sqft"), 2)
    ) \
    .withColumn(
        "property_age",
        lit(2024) - col("year_built")
    ) \
    .withColumn(
        "price_per_bed",
        spark_round(col("list_price") / col("bedrooms"), 0)
    )

print("=== Listings with Computed Columns ===")
enriched_df.select(
    "address", "list_price", "sqft",
    "price_per_sqft", "year_built", "property_age",
    "bedrooms", "price_per_bed"
).show(8)

print(f"\nNew column count: {len(enriched_df.columns)} (was {len(df.columns)})")

=== Listings with Computed Columns ===


+-------------+----------+----+--------------+----------+------------+--------+-------------+
|      address|list_price|sqft|price_per_sqft|year_built|property_age|bedrooms|price_per_bed|
+-------------+----------+----+--------------+----------+------------+--------+-------------+
|7370 River Rd| 1268597.0|NULL|          NULL|      NULL|        NULL|       5|     253719.0|
|960 Park Blvd|  835503.0|NULL|          NULL|      NULL|        NULL|    NULL|         NULL|
| 5291 Hill Dr|  487283.0|NULL|          NULL|      NULL|        NULL|       3|     162428.0|
| 5834 Oak Ave|  733117.0|NULL|          NULL|      NULL|        NULL|       3|     244372.0|
|566 Park Blvd|  951151.0|NULL|          NULL|      NULL|        NULL|       4|     237788.0|
| 5678 Hill Dr|  550782.0|NULL|          NULL|      NULL|        NULL|       5|     110156.0|
| 8422 Hill Dr|  174042.0|NULL|          NULL|      NULL|        NULL|       3|      58014.0|
| 869 River Rd|      NULL|NULL|          NULL|      NULL|   

## Sorting Results - Best First

After filtering, buyers want to see the best options first.
`orderBy()` (also available as `sort()`) sorts by one or more columns.

```python
df.orderBy("list_price")                          # ascending (default)
df.orderBy(col("list_price").asc())               # explicit ascending
df.orderBy(col("list_price").desc())              # descending
df.orderBy(col("neighborhood").asc(),             # multiple columns
           col("list_price").desc())
```


In [6]:
print("=== Cheapest Listings First ===")
df.select("address", "neighborhood", "list_price", "bedrooms", "sqft") \
  .orderBy(col("list_price").asc()) \
  .show(5)

print("=== Most Expensive Listings First ===")
df.select("address", "neighborhood", "list_price", "bedrooms", "sqft") \
  .orderBy(col("list_price").desc()) \
  .show(5)

print("=== Neighborhood A-Z, then Price Low-to-High ===")
df.select("address", "neighborhood", "list_price", "bedrooms") \
  .orderBy(
      col("neighborhood").asc(),
      col("list_price").asc()
  ) \
  .show(8)

=== Cheapest Listings First ===


+--------------+------------+----------+--------+----+
|       address|neighborhood|list_price|bedrooms|sqft|
+--------------+------------+----------+--------+----+
|7485 Park Blvd|     Seaside|      NULL|       1|NULL|
|1584 Park Blvd|    Parkview|      NULL|       2|NULL|
|6768 Park Blvd|  Greenfield|      NULL|       4|NULL|
|  7674 Hill Dr|  Greenfield|      NULL|       2|NULL|
|  869 River Rd|    Parkview|      NULL|       3|NULL|
+--------------+------------+----------+--------+----+
only showing top 5 rows
=== Most Expensive Listings First ===


+-------------+------------+----------+--------+----+
|      address|neighborhood|list_price|bedrooms|sqft|
+-------------+------------+----------+--------+----+
| 4568 Oak Ave|     Midtown| 1497758.0|       2|NULL|
| 5434 Hill Dr|  Greenfield| 1497203.0|       2|NULL|
|7214 River Rd|    Downtown| 1496853.0|       4|NULL|
|6284 River Rd|     Seaside| 1496419.0|       4|NULL|
| 5636 Oak Ave|     Seaside| 1491293.0|       5|NULL|
+-------------+------------+----------+--------+----+
only showing top 5 rows
=== Neighborhood A-Z, then Price Low-to-High ===


+--------------+------------+----------+--------+
|       address|neighborhood|list_price|bedrooms|
+--------------+------------+----------+--------+
|  1431 Main St|        NULL|  155730.0|       4|
|  2589 Main St|        NULL|  335550.0|    NULL|
|7041 Park Blvd|        NULL|  351904.0|       3|
| 9770 River Rd|        NULL|  386495.0|       3|
|7626 Park Blvd|        NULL|  424349.0|       3|
|8107 Park Blvd|        NULL|  536213.0|       2|
|  4737 Oak Ave|        NULL|  558213.0|       2|
|  4306 Oak Ave|        NULL|  582206.0|       4|
+--------------+------------+----------+--------+
only showing top 8 rows


In [7]:
# Practical pipeline: Top 10 best-value ACTIVE listings (3+ beds)
# Best value = lowest price-per-sqft

print("=" * 60)
print("   TOP 10 BEST VALUE ACTIVE LISTINGS (3+ Bedrooms)")
print("=" * 60)

top_value = df \
    .filter(
        (col("status") == "Active") &
        (col("bedrooms") >= 3) &
        col("sqft").isNotNull() &
        (col("sqft") > 0)
    ) \
    .withColumn(
        "price_per_sqft",
        spark_round(col("list_price") / col("sqft"), 2)
    ) \
    .select(
        "address",
        "neighborhood",
        col("list_price").alias("asking_price"),
        "bedrooms", "bathrooms", "sqft", "price_per_sqft"
    ) \
    .orderBy(col("price_per_sqft").asc()) \
    .limit(10)

top_value.show(truncate=False)

   TOP 10 BEST VALUE ACTIVE LISTINGS (3+ Bedrooms)


+-------+------------+------------+--------+---------+----+--------------+
|address|neighborhood|asking_price|bedrooms|bathrooms|sqft|price_per_sqft|
+-------+------------+------------+--------+---------+----+--------------+
+-------+------------+------------+--------+---------+----+--------------+



In [8]:
# distinct() returns unique rows across ALL columns
# dropDuplicates([cols]) checks only specified columns for uniqueness

print("=== Distinct Neighborhoods ===")
unique_hoods = df.select("neighborhood").distinct().orderBy("neighborhood")
unique_hoods.show()
print(f"Unique neighborhoods: {unique_hoods.count()}")

print("\n=== Distinct Property Types ===")
df.select("property_type").distinct().orderBy("property_type").show()

print("\n=== Distinct Status Values ===")
df.select("status").distinct().show()

# dropDuplicates on multiple columns
one_per_combo = df.dropDuplicates(["neighborhood", "property_type"])
print(f"Total listings:                            {df.count()}")
print(f"After dropDuplicates(neighborhood, type):  {one_per_combo.count()}")

=== Distinct Neighborhoods ===


+------------+
|neighborhood|
+------------+
|        NULL|
|    Downtown|
|  Greenfield|
|    Hillside|
|     Midtown|
|     Oakwood|
|    Parkview|
|   Riverside|
|     Seaside|
+------------+



Unique neighborhoods: 9

=== Distinct Property Types ===


+-------------+
|property_type|
+-------------+
|         NULL|
|    Apartment|
|        Condo|
|        House|
|    Townhouse|
|        Villa|
+-------------+


=== Distinct Status Values ===


+----------+
|    status|
+----------+
|   Pending|
|Off Market|
|      Sold|
|  For Sale|
|      NULL|
+----------+

Total listings:                            510


After dropDuplicates(neighborhood, type):  53


## Lesson 3 Wrap-Up

You have learned how to navigate a Spark DataFrame like a seasoned buyer's agent.

### Key Takeaways

| Operation | Method | Description |
|-----------|--------|-------------|
| Pick columns | `select()` | Returns only specified columns |
| Filter rows | `filter()` / `where()` | Keeps rows matching conditions |
| Rename columns | `alias()` / `withColumnRenamed()` | Cleaner column names |
| Add new columns | `withColumn()` | Computed or derived columns |
| Sort results | `orderBy()` / `sort()` | Ascending or descending |
| Limit rows | `limit(n)` | Top N rows |
| Remove duplicates | `distinct()` / `dropDuplicates()` | Unique rows |

### The Lazy Evaluation Pipeline Pattern

In Spark, operations are **lazy** -- they do not execute until an **action**
(like `show()`, `count()`, or `write()`) is called. Chain transformations freely:

```python
result = df \
    .filter(condition) \
    .withColumn("new_col", expression) \
    .select("col_a", "col_b", "new_col") \
    .orderBy("new_col") \
    .limit(10)
```

---

**Next: Lesson 4 - Cleaning the Books**

Real data is messy -- nulls, wrong types, inconsistent values.
Before you can trust your analysis, you need to clean the data.
Time to put on your inspector's hat.
